# DSFMTGen — dSFMT521 equidistribution

**dSFMT** (double-precision SFMT, Saito 2009) emits 52-bit mantissas
directly, skipping the 64→52-bit conversion the application would
otherwise do. Structurally similar to SFMT (SIMD-aware, non-primitive
$\chi_f$), but $L = 52$ to match the IEEE-754 mantissa width.

This notebook uses dSFMT521 (the smallest variant). See
[generators/DSFMTGen.md](../../generators/DSFMTGen.md).


## Imports


In [ ]:
# stamp:auto-generated
from regpoly.core.generator import Generator
from regpoly.core.combination import Combination
from regpoly.core.transformation import Transformation
from regpoly.analyses.equidistribution_test import (
    EquidistributionTest,
    METHOD_MATRICIAL, METHOD_HARASE,
    METHOD_NOTPRIMITIVE, METHOD_SIMD_NOTPRIMITIVE,
)

INT_MAX = 2**31 - 1


## Construct the generator — _Saito (2009)_


In [ ]:
# dSFMT521 parameters (Saito 2009 — supplementary table).
gen = Generator.create("DSFMTGen", L=52,
    mexp=521, pos1=3, sl1=25,
    msk1=0x000FBFEFFF77EFFF, msk2=0x000FFEEBFBDFBFDF,
    fix1=0xCFB393D661638469, fix2=0xC166867883AE2ADB,
    pcv1=0xCCAA588000000000, pcv2=0x0000000000000001)
print(gen.display())


## Wrap in a `Combination`


In [ ]:
# Wrap the generator in a single-component Combination. The
# Combination is the live object the search loop iterates and the
# equidistribution test consumes.
comb = Combination(J=1, Lmax=gen.L)
comb.components[0].add_gen(gen)
comb.reset()
print(f"k_g = {comb.k_g}, L = {comb.L}")


## Equidistribution test

Same SIMD-aware path as SFMT.


In [ ]:
# Build the equidistribution test and run it. We cap `delta` at
# INT_MAX so nothing is rejected — we just want the score.
test = EquidistributionTest(
    L=gen.L,
    delta=[INT_MAX] * (gen.L + 1),
    mse=INT_MAX,
    method=METHOD_SIMD_NOTPRIMITIVE,
)
result = test.run(comb)

print(f"SE (Σ gaps)   = {result.se}")
print(f"verified      = {result.verified}")
print(f"first 10 gaps = {[result.ecart[i] for i in range(1, min(11, gen.L + 1))]}")


## Catalog entry

The published version of this parameter set lives in the REGPOLY catalog under `library_id = "dsfmt521"`. To load it programmatically without hard-coding parameters:

```python
from regpoly.library import Catalog
cat = Catalog('docs/library')
cat.load()
_, entry = cat.generator('dsfmt521')
# entry.components[0] carries the same params as constructed above
```
